In [5]:
import fitz  # PyMuPDF
import os

pdf_folder = "athletic/bibs"
output_folder = "athletic/bibs"
os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(pdf_folder):
    if file.lower().endswith(".pdf"):
        pdf_path = os.path.join(pdf_folder, file)
        doc = fitz.open(pdf_path)
        base_name = os.path.splitext(file)[0]

        for page_number, page in enumerate(doc):
            pix = page.get_pixmap(dpi=300)
            output_path = os.path.join(output_folder, f"{base_name}.png")
            pix.save(output_path)

        print(f"✅ {file} converti ({len(doc)} pages)")


✅ 2022_velo_4S.pdf converti (1 pages)
✅ 2023_cap_20km_Lausanne.pdf converti (1 pages)
✅ 2023_cap_escalade_duc.pdf converti (1 pages)
✅ 2023_cap_vortex.pdf converti (1 pages)
✅ 2023_velo_EDT.pdf converti (1 pages)
✅ 2024_cap_20km_Lausanne.pdf converti (1 pages)
✅ 2024_cap_escalade_elite.pdf converti (1 pages)
✅ 2024_cap_vortex.pdf converti (1 pages)
✅ 2024_ski_Engadin.pdf converti (1 pages)
✅ 2024_tri_Huez.pdf converti (1 pages)
✅ 2024_tri_Rumilly.pdf converti (1 pages)
✅ 2024_tri_Troyes.pdf converti (1 pages)
✅ 2025_cap_gva_marathon.pdf converti (1 pages)
✅ 2025_cap_Saint_Genese.pdf converti (1 pages)
✅ 2025_trail_Gets.pdf converti (1 pages)
✅ 2025_trail_hiver_Gets.pdf converti (1 pages)
✅ 2025_tri_Rumilly.pdf converti (1 pages)
✅ 2025_velo_JPP.pdf converti (1 pages)


In [ ]:
# import cv2 
# img = cv2.imread('projects/images/watchmaking2.jpg')
# cv2.imwrite('projects/images/watchmaking2.png', img, [cv2.IMWRITE_PNG_COMPRESSION, 0])

True

# Script d'Optimisation des Images Athletic

Ce script va compresser et optimiser toutes les images du dossier athletic pour améliorer les performances du site.

In [13]:
from PIL import Image
import os
from pathlib import Path

def optimize_image(input_path, output_path, max_width=1920, quality=85):
    """
    Optimise une image en la redimensionnant et la compressant
    
    Args:
        input_path: Chemin de l'image source
        output_path: Chemin de l'image optimisée
        max_width: Largeur maximale (défaut: 1920px)
        quality: Qualité de compression JPEG (défaut: 85)
    """
    try:
        # Ouvrir l'image
        img = Image.open(input_path)
        
        # Obtenir la taille originale
        original_size = os.path.getsize(input_path) / 1024  # en KB
        
        # Redimensionner si l'image est trop grande
        if img.width > max_width:
            ratio = max_width / img.width
            new_height = int(img.height * ratio)
            img = img.resize((max_width, new_height), Image.Resampling.LANCZOS)
        
        # Convertir RGBA en RGB si nécessaire (pour les PNG avec transparence)
        if img.mode == 'RGBA':
            # Créer un fond blanc
            background = Image.new('RGB', img.size, (255, 255, 255))
            background.paste(img, mask=img.split()[3])  # 3 est le canal alpha
            img = background
        elif img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Sauvegarder l'image optimisée
        img.save(output_path, 'JPEG', quality=quality, optimize=True)
        
        # Obtenir la nouvelle taille
        new_size = os.path.getsize(output_path) / 1024  # en KB
        
        # Calculer la réduction
        reduction = ((original_size - new_size) / original_size) * 100
        
        return {
            'success': True,
            'original_size': original_size,
            'new_size': new_size,
            'reduction': reduction
        }
    
    except Exception as e:
        return {
            'success': False,
            'error': str(e)
        }

print("✅ Fonction optimize_image créée")

✅ Fonction optimize_image créée


In [17]:
print("\n📂 OPTIMISATION DES IMAGES PROJECTS")
print("=" * 50)

projects_folder = "personal-page/images"
backup_folder = "athletic/backups_originaux/images"
projects_stats = {
    'total': 0,
    'optimized': 0,
    'failed': 0,
    'total_original_size': 0,
    'total_new_size': 0
}

# Créer le dossier de backup pour projects
os.makedirs(backup_folder, exist_ok=True)

for filename in os.listdir(projects_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        input_path = os.path.join(projects_folder, filename)
        
        # # Créer un backup
        # backup_path = os.path.join(f"{backup_folder}/projects_images", filename)
        # if not os.path.exists(backup_path):
        #     import shutil
        #     shutil.copy2(input_path, backup_path)
        
        # Convertir en JPG
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}.jpg"
        output_path = os.path.join(projects_folder, output_filename)
        
        # Optimiser avec qualité 85 et max width 1920px
        result = optimize_image(input_path, output_path, max_width=1920, quality=85)
        
        projects_stats['total'] += 1
        
        if result['success']:
            projects_stats['optimized'] += 1
            projects_stats['total_original_size'] += result['original_size']
            projects_stats['total_new_size'] += result['new_size']
            
            print(f"✅ {filename}")
            print(f"   {result['original_size']:.1f} KB → {result['new_size']:.1f} KB (-{result['reduction']:.1f}%)")
            
            # Supprimer l'ancien si différent format
            if filename != output_filename and os.path.exists(output_path):
                os.remove(input_path)
                print(f"   🗑️ Ancien fichier supprimé")
        else:
            projects_stats['failed'] += 1
            print(f"❌ {filename}: {result['error']}")

print("\n" + "=" * 50)
print(f"📊 RÉSUMÉ IMAGES PROJECTS:")
print(f"   Total traité: {projects_stats['total']}")
print(f"   Optimisés: {projects_stats['optimized']}")
print(f"   Échecs: {projects_stats['failed']}")
print(f"   Taille originale: {projects_stats['total_original_size']:.1f} KB ({projects_stats['total_original_size']/1024:.1f} MB)")
print(f"   Nouvelle taille: {projects_stats['total_new_size']:.1f} KB ({projects_stats['total_new_size']/1024:.1f} MB)")
reduction_projects = ((projects_stats['total_original_size'] - projects_stats['total_new_size']) / projects_stats['total_original_size'] * 100) if projects_stats['total_original_size'] > 0 else 0
print(f"   Réduction totale: {reduction_projects:.1f}%")


📂 OPTIMISATION DES IMAGES PROJECTS
✅ climber.jpg
   353.0 KB → 353.0 KB (-0.0%)
✅ cyclist.jpg
   211.7 KB → 211.7 KB (--0.0%)
✅ cyclist2.jpg
   293.8 KB → 293.8 KB (--0.0%)
✅ EPFLRT_team.jpg
   169.0 KB → 169.1 KB (--0.0%)
✅ hero.jpg
   159.1 KB → 159.2 KB (--0.1%)
✅ runner.jpg
   185.0 KB → 185.0 KB (--0.0%)
✅ scara.jpg
   227.5 KB → 227.5 KB (-0.0%)
✅ skier.jpg
   323.8 KB → 323.8 KB (--0.0%)
✅ skier2.jpg
   330.2 KB → 330.3 KB (--0.0%)
✅ wales.jpg
   233.2 KB → 239.4 KB (--2.7%)

📊 RÉSUMÉ IMAGES PROJECTS:
   Total traité: 10
   Optimisés: 10
   Échecs: 0
   Taille originale: 2486.3 KB (2.4 MB)
   Nouvelle taille: 2492.7 KB (2.4 MB)
   Réduction totale: -0.3%
